# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the **FAIR^2** dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant/) library. The notebook follows a structured workflow: loading metadata, inspecting record sets and fields, extracting tabular data, performing exploratory analyses, and visualizing results.

### Dataset Source

The dataset is published as a Croissant schema and accessed via the following URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Install mlcroissant if needed
!pip install --quiet mlcroissant

## 1. Data Loading

Load dataset metadata and explore the schema.

In [ ]:
import mlcroissant as mlc
import pandas as pd

croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load Dataset schema
dataset = mlc.Dataset(croissant_url)

# Access metadata attributes
meta = dataset.metadata
print(f"Dataset name: {meta.name}")
print(f"  Croissant schema version: {meta.conformsTo}")
print(f"  Version: {getattr(meta, 'version', None)}")
print(f"  Identifier: {getattr(meta, 'identifier', None)}")
print(f"  Citation: {getattr(meta, 'citeAs', None)}\n")
print(f"Description: {meta.description}\n")
print(f"Fields: {getattr(meta, 'keywords', [])}")

## 2. Data Overview

The Croissant schema organizes data as *RecordSets*. Let's enumerate the available RecordSets, their `@id`, and fields. 

We will use the `dataset.record_sets` API to list them. All references use the `@id` value for precision.

In [ ]:
# List all RecordSets and their Fields/Columns by @id
print("Available Record Sets (by @id):")
for record_set in dataset.record_sets:
    print(f"- RecordSet @id: {record_set.id}")
    print(f"  Name: {record_set.name}")
    print("  Fields/Columns:")
    # Some schemas use .fields, some .columns
    if hasattr(record_set, 'fields') and record_set.fields:
        for fld in record_set.fields:
            print(f"    - Field @id: {fld.id}, name: {fld.name}, dataType: {fld.data_type}")
    if hasattr(record_set, 'columns') and record_set.columns:
        for col in record_set.columns:
            print(f"    - Column @id: {col.id}, name: {col.name}, dataType: {col.data_type}")
    print()

## 3. Data Extraction

Let's select a tabular RecordSet and read its records into a DataFrame. We will use the RecordSet and field `@id`s identified above.

Below, we load *all record sets* into dataframes for analysis and inspect the first few columns and rows.

In [ ]:
# Identify all RecordSet @id (dynamically, so notebook adapts to dataset)
record_set_ids = [rs.id for rs in dataset.record_sets]

dataframes = {}
for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} records from RecordSet @id '{rs_id}'; columns: {list(df.columns)}")
    except Exception as e:
        print(f"Could not load records for RecordSet @id '{rs_id}': {e}")

if dataframes:
    example_rs_id = record_set_ids[0]
    print(f"\n-----\nColumns for example RecordSet @id: {example_rs_id}\n{dataframes[example_rs_id].columns.tolist()}")
    dataframes[example_rs_id].head()

## 4. Exploratory Data Analysis (EDA)

Let's perform basic EDA: filtering a numeric field, normalizing it, and grouping by a categorical field, if present.

In [ ]:
# Select a RecordSet for EDA
import numpy as np
rs_id = example_rs_id
df = dataframes[rs_id]

# Try to guess a numeric field for demonstration
numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
if not numeric_candidates:
    # Try to coerce object columns to numeric
    for c in df.columns:
        try:
            df[c] = pd.to_numeric(df[c])
        except:
            continue
    numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()

if numeric_candidates:
    numeric_field = numeric_candidates[0]
    print(f"Using numeric field: {numeric_field}\n")
    threshold = df[numeric_field].mean() if not np.isnan(df[numeric_field].mean()) else 10
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    print(filtered_df.head())

    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a key attribute
    group_candidates = [c for c in df.columns if df[c].nunique() < len(df)//4 and c != numeric_field]
    if group_candidates:
        group_field = group_candidates[0]
        print(f"\nGrouping filtered results by '{group_field}':")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(grouped_df.head())
    else:
        print("No suitable categorical group field found.")
else:
    print("No numeric columns found for EDA.")

## 5. Visualization

Visualize distributions or relationships (histograms, boxplots, scatter) for numeric fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_candidates:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If grouping field exists, boxplot
    if 'group_field' in locals():
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()

## 6. Conclusion

In this notebook, we explored a mass spectrometry dataset of human extracellular vesicle (EV) proteins using the `mlcroissant` library. After loading and examining the Croissant metadata, we extracted available record sets, loaded their records, performed basic EDA (including filtering and normalization of numeric fields), and visualized data distributions.

**Next steps:** This dataset supports further analysis such as differential protein expression studies, peptide coverage analysis, and integration into downstream bioinformatics pipelines. For detailed field mapping, reference the metadata output in Section 2.
